# Practical Application 2: What Drives the Price of a Car?
**Framework:** CRISP-DM  |  **Audience:** Used-car dealership management

Objective: identify vehicle attributes associated with used-vehicle listing prices and translate the results into acquisition and appraisal guidance.

Dataset: 426,880 supplied listings. Place `vehicles.csv` in `../data/` before running. The raw dataset is excluded from Git.

## 1. Business understanding
Dealers balance acquisition cost, resale demand, inventory turn, and margin. This analysis examines age, mileage, manufacturer, condition, fuel, title status, transmission, drivetrain, and body type.

**Important:** the target is advertised listing price, not verified transaction price. Observational differences are associations, not causal effects.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
df = pd.read_csv('../data/vehicles.csv')
print('Initial shape:', df.shape)
display(df.head())
display(pd.DataFrame({'missing_count': df.isna().sum(), 'missing_pct': (df.isna().mean()*100).round(2)}).sort_values('missing_count', ascending=False))

## 2. Data preparation
Drop identifiers and the highly incomplete `size` field. Filter price to $1,500–$80,000, year to 1995–2022, and odometer to 1,000–250,000 miles. Derive age using 2022 as the dataset reference year, then use complete cases for the selected modeling fields.

In [ ]:
clean_df = df.drop(columns=['id', 'VIN', 'size'], errors='ignore').copy()
clean_df = clean_df[clean_df.price.between(1500, 80000)]
clean_df = clean_df[clean_df.year.between(1995, 2022)]
clean_df = clean_df[clean_df.odometer.between(1000, 250000)]
clean_df['vehicle_age'] = 2022 - clean_df['year']
features = ['price','vehicle_age','odometer','manufacturer','condition','cylinders','fuel','title_status','transmission','drive','type']
clean_df = clean_df[features].dropna().copy()
print('Cleaned complete-case shape:', clean_df.shape)
print('Rows retained:', f'{len(clean_df):,}', f'({len(clean_df)/len(df):.1%})')
display(clean_df.describe(include='all').T)

## 3. Exploratory analysis
The plots examine price distribution and its relationship with age and mileage, followed by unadjusted comparisons across body type and drivetrain.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(clean_df.price, bins=40, kde=True, ax=axes[0]); axes[0].set_title('Price distribution')
sample_viz = clean_df.sample(min(5000, len(clean_df)), random_state=42)
sns.scatterplot(data=sample_viz, x='vehicle_age', y='price', alpha=.3, ax=axes[1]); axes[1].set_title('Price vs age')
sns.scatterplot(data=sample_viz, x='odometer', y='price', alpha=.3, ax=axes[2]); axes[2].set_title('Price vs odometer')
plt.tight_layout(); plt.savefig('../images/continuous_distributions.png', dpi=160); plt.show()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(data=clean_df, x='type', y='price', order=clean_df.groupby('type').price.median().sort_values(ascending=False).index, showfliers=False, ax=axes[0]); axes[0].tick_params(axis='x', rotation=45); axes[0].set_title('Price by body type')
sns.boxplot(data=clean_df, x='drive', y='price', order=clean_df.groupby('drive').price.median().sort_values(ascending=False).index, showfliers=False, ax=axes[1]); axes[1].set_title('Price by drivetrain')
plt.tight_layout(); plt.savefig('../images/categorical_comparisons.png', dpi=160); plt.show()
for col in ['type','drive','fuel','title_status']:
    print('\nMedian price by', col)
    display(clean_df.groupby(col).price.agg(['count','median']).sort_values('median', ascending=False))

## 4. Modeling and evaluation
Compare Linear Regression, Ridge, Lasso, and Decision Tree using a fixed random sample of up to 40,000 cleaned records, a 75/25 train/test split, and five-fold cross-validation. Numeric features are standardized and categorical features one-hot encoded. RMSE is reported in dollars; R² is the proportion of test variance explained.

In [ ]:
X = clean_df.drop(columns='price'); y = clean_df.price
sample = clean_df.sample(n=min(40000, len(clean_df)), random_state=42)
X_train, X_test, y_train, y_test = train_test_split(sample.drop(columns='price'), sample.price, test_size=.25, random_state=42)
num_cols = ['vehicle_age','odometer']
cat_cols = ['manufacturer','condition','cylinders','fuel','title_status','transmission','drive','type']
prep = ColumnTransformer([('num', StandardScaler(), num_cols), ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)])
def evaluate(pipe, grid, name):
    search = GridSearchCV(pipe, grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
    search.fit(X_train, y_train); model = search.best_estimator_
    train_pred, test_pred = model.predict(X_train), model.predict(X_test)
    return {'Model': name, 'Best_Params': search.best_params_, 'Train_RMSE': mean_squared_error(y_train, train_pred)**.5, 'Test_RMSE': mean_squared_error(y_test, test_pred)**.5, 'Train_R2': r2_score(y_train, train_pred), 'Test_R2': r2_score(y_test, test_pred), 'Model_Object': model}
results = [
 evaluate(Pipeline([('prep',prep),('reg',LinearRegression())]), {}, 'Linear Regression'),
 evaluate(Pipeline([('prep',prep),('reg',Ridge())]), {'reg__alpha':[.1,1,10,100]}, 'Ridge Regression'),
 evaluate(Pipeline([('prep',prep),('reg',Lasso(max_iter=3000))]), {'reg__alpha':[.1,1,10]}, 'Lasso Regression'),
 evaluate(Pipeline([('prep',prep),('reg',DecisionTreeRegressor(random_state=42))]), {'reg__max_depth':[6,10,14], 'reg__min_samples_split':[20,50]}, 'Decision Tree')]
summary_df = pd.DataFrame([{k:v for k,v in r.items() if k != 'Model_Object'} for r in results])
display(summary_df.sort_values('Test_RMSE'))

## 5. Model interpretation
Ridge coefficients are directional conditional associations. Numeric coefficients represent approximate price changes per standard deviation; categorical coefficients are contrasts against dropped reference categories. They are not causal effects.

In [ ]:
ridge = next(r['Model_Object'] for r in results if r['Model'] == 'Ridge Regression')
encoder = ridge.named_steps['prep'].named_transformers_['cat']
names = num_cols + list(encoder.get_feature_names_out(cat_cols))
coef_df = pd.DataFrame({'Feature': names, 'Coefficient': ridge.named_steps['reg'].coef_})
top = coef_df.assign(abs_value=lambda d: d.Coefficient.abs()).nlargest(15, 'abs_value').sort_values('Coefficient')
display(top)
plt.figure(figsize=(12,7)); plt.barh(top.Feature, top.Coefficient); plt.title('Top 15 Ridge price coefficients'); plt.xlabel('Coefficient'); plt.tight_layout(); plt.savefig('../images/feature_importance.png', dpi=160); plt.show()

## 6. Business interpretation, recommendations, and limitations
- Use age and odometer jointly in appraisals; avoid a single mileage cutoff across segments.
- Consider late-model pickups, trucks, and 4WD vehicles where local demand and acquisition economics justify them.
- Make title verification mandatory; non-clean titles require additional legal, safety, inspection, and reconditioning diligence.
- Use model estimates as a benchmark alongside local comparables, inspection results, reconditioning costs, and expected days-to-sale.
- Monitor prediction error by region, body type, manufacturer, and price band before operational deployment.

**Limitations:** asking prices are not transaction prices; complete-case filtering retains 110,429 of 426,880 rows and may introduce selection bias; age is referenced to 2022; group medians are unadjusted; a random split does not test temporal or geographic generalization. Validate against realized sales before using predictions to set purchase limits.